# From Pixels to Production: CIFAR-10 Progressive Classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genieincodebottle/deep-learning-capstone/blob/main/notebooks/CIFAR10_Progressive.ipynb)

> Systematic experimentation from baseline CNN (~60%) to optimized ResNet (93%+).

**Important:** Use GPU runtime. Go to Runtime > Change runtime type > T4 GPU

## Setup

In [ ]:
!pip install torch torchvision -q
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## Step 1: Data Loading with Augmentation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms.v2 as T

transform_augmented = T.Compose([
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.RandomHorizontalFlip(p=0.5), T.RandomCrop(32, padding=4),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_test = T.Compose([
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_augmented)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)
print(f"Train: {len(trainset)}, Test: {len(testset)}, Classes: {trainset.classes}")

## Step 2: ResNet-Style Architecture

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + x)

class CIFAR10Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.stage1 = ResBlock(64)
        self.pool1 = nn.MaxPool2d(2)
        self.expand2 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU())
        self.stage2 = ResBlock(128)
        self.pool2 = nn.MaxPool2d(2)
        self.expand3 = nn.Sequential(nn.Conv2d(128, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU())
        self.stage3 = ResBlock(256)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(256, 10))
    def forward(self, x):
        x = self.pool1(self.stage1(self.stem(x)))
        x = self.pool2(self.stage2(self.expand2(x)))
        return self.head(self.stage3(self.expand3(x)))

model = CIFAR10Net()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 3: Training (50 epochs for Colab, use 200 for best results)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CIFAR10Net().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
use_amp = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
best_acc = 0.0

for epoch in range(50):
    model.train()
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = criterion(model(images), labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    scheduler.step()
    if (epoch + 1) % 5 == 0:
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in testloader:
                images, labels = images.to(device), labels.to(device)
                _, predicted = model(images).max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        acc = 100.0 * correct / total
        best_acc = max(best_acc, acc)
        print(f"Epoch {epoch+1:3d} | Loss: {running_loss/len(trainloader):.4f} | Acc: {acc:.1f}% | Best: {best_acc:.1f}%")

## Step 4: Per-Class Evaluation

In [ ]:
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = model(images).max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"{'Class':<15} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 43)
for i, name in enumerate(classes):
    mask = [j for j, l in enumerate(all_labels) if l == i]
    correct = sum(1 for j in mask if all_preds[j] == i)
    acc = 100.0 * correct / len(mask) if mask else 0
    print(f"{name:<15} {correct:>8} {len(mask):>8} {acc:>9.1f}%")

## Key Takeaways

- **Progressive approach**: Start simple, add one improvement at a time
- **Skip connections**: ResBlocks improve gradient flow for deeper networks
- **Mixed precision**: 2x training speed on GPU with minimal accuracy impact
- **Cosine annealing**: Smooth LR decay avoids plateau stalling
- **Per-class accuracy**: Reveals hardest classes (cat/dog confusion is common)